# FinTrust loan default prediction — requirements 2E–2J

This notebook executes the end-to-end workflow in `src/phase2e_2j.py`: ten-model five-fold comparison, fold-local imbalance strategies, tuning, threshold analysis, validation interpretation, final full-training refit, and required test/scenario predictions. Preprocessing remains inside sklearn pipelines. The test set is not used for model selection, tuning, or threshold selection.

In [ ]:
from pathlib import Path
import sys

search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_DIR = None
for root in search_roots:
    candidates = [root, *root.glob("*/task2_ml")]
    for candidate in candidates:
        if (candidate / "src" / "phase2e_2j.py").is_file():
            PROJECT_DIR = candidate.resolve()
            break
    if PROJECT_DIR is not None:
        break
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not locate the task2_ml project directory.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f"Project: {PROJECT_DIR}")

In [ ]:
from IPython.display import display
from src.phase2e_2j import run_complete_requirements

results = run_complete_requirements(PROJECT_DIR)
print("Selected model:", results["selected_model"])
print("Fixed threshold:", results["threshold"])
print("Test prediction rows:", len(results["test_predictions"]))
print("Scenario rows:", len(results["scenario_predictions"]))
print("Elapsed seconds:", round(results["elapsed_seconds"], 2))

In [ ]:
display(results["model_comparison"])
display(results["imbalance_comparison"])
display(results["tuning_results"])
display(results["scenario_predictions"])
display(results["scenario_explanations"])
print("All requested CSVs, reports, and visualizations are in outputs/.")

## Scenario result review (all 12 cases)

The saved pipeline uses a 0.11 cutoff. These probability-based flags are screening signals, not final lending decisions. The following review compares each result with the profile intent and highlights where a low score should **not** be treated as evidence that an input is valid or safe.

| Case | Probability | Flag | Does it look sensible? Why / what to check |
|---|---:|---|---|
| TC01 | 0.0087 | Non-default | Yes: the profile is described as prime, with high credit score and low obligations. |
| TC02 | 0.9298 | Default | Yes: low score, many recent late payments, and very high utilization agree with the high-risk profile. |
| TC03 | 0.1948 | Default | Plausible caution: a missing credit score raises uncertainty; this is not proof of default. Review the new-to-credit policy and missingness handling. |
| TC04 | 0.0154 | Non-default | Plausible if the monthly income text was normalized correctly; verify the parsed annual income rather than relying only on this low score. |
| TC05 | 0.0176 | Non-default | Pipeline compatibility is as expected for an unseen loan-purpose category. Low risk follows the other profile inputs, not evidence that the unseen category itself is safe. |
| TC06 | 0.0154 | Non-default | Pipeline handles the unseen branch value without a feature mismatch. Confirm branch availability and policy; the score is not a branch-specific validation. |
| TC07 | 0.0125 | Non-default | Dirty formats were accepted and normalized; manually inspect cleaned values before interpreting the low probability. |
| TC08 | 0.0408 | Non-default | Plausible given the strong remaining profile signals, but the high loan-to-income description warrants manual review despite being below threshold. |
| TC09 | 0.0226 | Non-default | Compared with TC08, added collateral/co-signer corresponds to a lower probability. This is a useful directional check, not a causal conclusion. |
| TC10 | 0.0188 | Non-default | The pipeline accepts multiple missing fields, but a low score with substantial missingness should be referred for data-quality/manual review. |
| TC11 | 0.0151 | Non-default | **Not a sensible approval signal:** the profile contains a negative loan amount. It is flagged as invalid input; correct/reject the record rather than interpret its imputed score. |
| TC12 | 0.0108 | Non-default | Plausible for the described prime profile; confirm the cleaned profile values if using it as a sanity case. |

All 12 rows and probabilities are in `outputs/scenario_predictions.csv`; detailed local explanations for TC01, TC02, and TC08 are in `outputs/scenario_explanations.md`. The invalid TC11 case demonstrates why input validation and human review remain necessary.